# Module 01: Macro Delay Forecasting (Prophet)

Predicts aggregate flight-delay risk over time so the twin's ambient risk baseline reflects real seasonal/day-of-week patterns instead of pure randomness. Trains on a public BTS-derived delay-cause dataset and exports a Prophet model consumed by `core/models.py`.

In [ ]:
!pip install -q prophet pandas numpy joblib requests


## 1. Fetch a public airline on-time performance dataset

We use a mirror of the U.S. BTS "Airline Delay Cause" monthly aggregate
(carrier x airport x month), hosted as a plain CSV on GitHub so it needs no
Kaggle/BTS login and works with a single `requests.get`. If the mirror is
ever unreachable, we fall back to a seasonally-realistic synthetic series so
this cell always succeeds.

In [ ]:
import io
import numpy as np
import pandas as pd
import requests

DATA_URL = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Airline%20Delay.csv"

def load_delay_dataframe():
    resp = requests.get(DATA_URL, timeout=20)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    df.columns = [c.strip().lower() for c in df.columns]
    return df

try:
    raw = load_delay_dataframe()
    print(f"Downloaded real dataset: {raw.shape[0]} rows, columns: {list(raw.columns)[:12]}...")

    # Auto-detect a date-like column instead of hard-coding schema, since
    # public mirrors occasionally rename columns.
    date_col = next((c for c in raw.columns if "date" in c or c in ("fl_date", "year")), None)
    if date_col is None:
        raise ValueError("Could not auto-detect a date column in the mirror - using synthetic series.")

    if date_col == "year" and "month" in raw.columns:
        raw["ds"] = pd.to_datetime(dict(year=raw["year"], month=raw["month"], day=1))
    else:
        raw["ds"] = pd.to_datetime(raw[date_col], errors="coerce")

    # Prefer average delay *per flight* (arr_delay / arr_flights) over the raw
    # aggregate so the scale is "minutes of delay per flight" - the same units
    # core/models.py expects when normalising Prophet's forecast into a risk score.
    if "arr_delay" in raw.columns and "arr_flights" in raw.columns:
        raw["y"] = pd.to_numeric(raw["arr_delay"], errors="coerce") / raw["arr_flights"].replace(0, np.nan)
    else:
        delay_col = next((c for c in raw.columns if "delay" in c and "id" not in c), None)
        if delay_col is None:
            raise ValueError("Could not auto-detect a delay column in the mirror - using synthetic series.")
        raw["y"] = pd.to_numeric(raw[delay_col], errors="coerce")
    series = raw.dropna(subset=["ds", "y"]).groupby("ds", as_index=False)["y"].mean().sort_values("ds")

    if len(series) < 30:
        raise ValueError("Aggregated series too short after cleaning - using synthetic series.")

    df = series[["ds", "y"]].reset_index(drop=True)
    data_source = "YBI-Foundation/Dataset Airline Delay.csv (BTS-derived, live download)"

except Exception as exc:
    print(f"[fallback] Live dataset unavailable ({exc}). Generating a seasonally-realistic synthetic series instead.")
    dates = pd.date_range(start="2022-01-01", end="2025-01-01", freq="D")
    day_of_year = dates.dayofyear.values
    weekday = dates.weekday.values
    seasonal = 8 + 6 * np.sin(2 * np.pi * day_of_year / 365.0 - np.pi / 2)  # monsoon-season peak
    weekly = np.where(np.isin(weekday, [4, 5, 6]), 3.0, 0.0)               # weekend congestion bump
    noise = np.random.normal(0, 3.0, len(dates))
    y = np.clip(seasonal + weekly + noise, 0, None)
    df = pd.DataFrame({"ds": dates, "y": y})
    data_source = "synthetic (seasonal + weekly pattern, monsoon-season delay peak)"

print(f"Training series: {len(df)} points | source: {data_source}")
df.tail()


## 2. Train the Prophet forecasting model

Prophet's Stan backend occasionally fails to initialise on a fresh Colab
runtime (a known upstream quirk, unrelated to our data). We wrap training in
a try/except and fall back to plain day-of-week + seasonal-bucket averages
computed with pandas - deliberately **not** a custom Python class, since a
custom class pickled from a notebook's `__main__` often fails to unpickle
later inside `core/models.py`. Both branches export data `core/models.py`
can consume directly.

In [ ]:
try:
    from prophet import Prophet

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
    )
    model.fit(df)
    model_kind = "prophet"

    future = model.make_future_dataframe(periods=14, freq="D")
    forecast = model.predict(future)
    print("Forecast tail (next 14 days of aggregate delay-risk minutes):")
    print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(14).to_string(index=False))

except Exception as exc:
    print(f"[fallback] Prophet/Stan backend failed to initialise ({exc}).")
    print("Falling back to day-of-week + seasonal-bucket averages (plain data, no custom classes to pickle).")
    model_kind = "seasonal_naive"
    model = None

    naive = df.copy()
    naive["dow"] = naive["ds"].dt.dayofweek
    naive["doy_bucket"] = naive["ds"].dt.dayofyear // 14
    overall_mean = float(naive["y"].mean())
    dow_effect = (naive.groupby("dow")["y"].mean() - overall_mean).to_dict()
    season_effect = (naive.groupby("doy_bucket")["y"].mean() - overall_mean).to_dict()
    last_date = naive["ds"].max()

print(f"Model type: {model_kind}")


## 3. Export for the live twin

`core/models.py` loads this bundle and normalises the forecasted delay
minutes into a 0-1 macro delay-risk score consumed by
`ModelRegistry.predict_macro_delay_risk()`. The bundle's `model_kind` field
tells it whether `model` is a real Prophet object or whether to reconstruct
a forecast from the plain `dow_effect` / `season_effect` dictionaries.

In [ ]:
import joblib
from datetime import datetime, timezone

PKL_NAME = "01_macro_forecast.pkl"
bundle = {
    "model_kind": model_kind,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "data_source": data_source,
    "module": "01_macro_forecast",
}
if model_kind == "prophet":
    bundle["model"] = model
else:
    bundle.update({
        "overall_mean": overall_mean,
        "dow_effect": dow_effect,
        "season_effect": season_effect,
        "last_date": last_date,
    })

joblib.dump(bundle, PKL_NAME)
print(f"Saved {PKL_NAME} (model_kind={model_kind})")


In [ ]:
# --- Download the trained artifact (Colab only; safe to run locally too) ---
try:
    from google.colab import files
    files.download(PKL_NAME)
    print(f"Downloading {PKL_NAME} ... move it into core/models/ on your machine.")
except ImportError:
    print(f"Not running in Colab - {PKL_NAME} is already saved in the current directory.")
    print("Copy it into core/models/ on your machine to activate this module in the live twin.")
